# Vehicle Data: Cleaning, EDA, Modeling & Explainability

End-to-end pipeline: data cleaning, feature engineering, model tuning (Linear Regression, Random Forest, XGBoost), evaluation, rare-segment analysis, feature importance, and SHAP explainability.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import time

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    KFold,
    cross_val_score
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

RANDOM_STATE = 42

## 2. Load Dataset

In [ ]:
df = pd.read_excel('Vehicle_Data.xlsx')
df.columns = df.columns.str.strip()

print('Original shape:', df.shape)
df.head()

## 3. Initial Checks

In [ ]:
print('\nMissing values:')
print(df.isna().sum())

print('\nExact duplicate rows:', df.duplicated().sum())

print('\nData types:')
print(df.dtypes)

## 4. Data Cleaning

### Remove Duplicates (Hybrid)

In [ ]:
# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Normalize key fields before duplicate detection
for col in ['Make', 'Model']:
    df[col] = (
        df[col].astype(str)
        .str.strip()
        .str.lower()
    )

# Remove duplicate listings
df = df.drop_duplicates(
    subset=['Make', 'Model', 'YOM', 'Mileage', 'Price'],
    keep='first'
).copy()

print('Shape after duplicate removal:', df.shape)

### Clean Price

In [ ]:
df['Price'] = (
    df['Price'].astype(str)
    .str.replace('KSh', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

### Clean Mileage

In [ ]:
df['Mileage'] = (
    df['Mileage'].astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
    .replace(['Unavailable', 'N/A', 'None', ''], np.nan)
)

df['Mileage'] = pd.to_numeric(df['Mileage'], errors='coerce')

### Clean Year

In [ ]:
df['YOM'] = pd.to_numeric(df['YOM'], errors='coerce')

### Remove Invalid Rows

In [ ]:
df = df.dropna(
    subset=['Make', 'Model', 'YOM', 'Price', 'Mileage']
).copy()

df = df[
    (df['YOM'].between(1960, 2026)) &
    (df['Price'] > 0) &
    (df['Mileage'] >= 0)
].copy()

print('Shape after removing invalid rows:', df.shape)

### Normalize Text Fields

In [ ]:
for col in ['Make', 'Model', 'Color', 'Used',
            'Transmission', 'Location']:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.title()
        )

### Handle Remaining Categorical Missing Values

In [ ]:
for col in ['Color', 'Used', 'Transmission', 'Location']:
    if col in df.columns:
        df[col] = df[col].replace(['', 'Nan'], np.nan)
        df[col] = df[col].fillna('Unknown')

print('\nRemaining missing values:')
remaining_missing = df.isna().sum()
print(remaining_missing[remaining_missing > 0])

## 5. Feature Engineering

In [ ]:
# County extraction
df['County'] = (
    df['Location']
    .str.split(',')
    .str[0]
    .str.strip()
    .str.title()
)

# Fill missing counties
df['County'] = df['County'].fillna(df['County'].mode()[0])

# Vehicle age
df['CarAge'] = 2026 - df['YOM']

# Mileage per year
df['MileagePerYear'] = df['Mileage'] / (df['CarAge'] + 1)

# Log transforms
df['LogPrice'] = np.log1p(df['Price'])
df['LogMileage'] = np.log1p(df['Mileage'])

df.head()

### Create Rare Segments

In [ ]:
brand_freq = df['Make'].value_counts(normalize=True)

rare_brands = brand_freq[brand_freq < 0.02].index.tolist()

df['RareSegment'] = np.where(
    df['Make'].isin(rare_brands),
    'Rare',
    'Common'
)

print(df['RareSegment'].value_counts())

## 6. Final Verification & Save

In [ ]:
remaining_dupes = df.duplicated(
    subset=['Make', 'Model', 'YOM', 'Mileage', 'Price']
).sum()

print('Remaining duplicates:', remaining_dupes)

assert remaining_dupes == 0
assert df.isna().sum().sum() == 0

print('\nFinal cleaned shape:', df.shape)

In [ ]:
df.to_csv('Vehicle_Data_cleaned.csv', index=False)
print('Cleaned dataset saved.')

## 7. Exploratory Data Analysis

### Distribution Plots

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
sns.histplot(df['Price'], bins=40, kde=True)
plt.title('Original Price')

plt.subplot(1,2,2)
sns.histplot(df['LogPrice'], bins=40, kde=True)
plt.title('Log Transformed Price')

plt.tight_layout()
plt.show()

### Correlation Matrix

In [ ]:
corr = df[['Price', 'Mileage', 'CarAge',
           'MileagePerYear']].corr()

plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap='Blues')
plt.title('Correlation Matrix')
plt.show()

## 8. Modeling

### Train/Test Split (Stratified)

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df['RareSegment']
)

print(train_df.shape, test_df.shape)

### Features and Target

In [ ]:
FEATURES = [
    'Make', 'Model', 'Color', 'Used',
    'Transmission', 'County',
    'CarAge', 'LogMileage', 'MileagePerYear'
]

TARGET = 'LogPrice'

X_train = train_df[FEATURES]
X_test = test_df[FEATURES]

y_train = train_df[TARGET]
y_test = test_df[TARGET]

### Preprocessing Pipeline

In [ ]:
NUMERIC = ['CarAge', 'LogMileage', 'MileagePerYear']
CATEGORICAL = ['Make', 'Model', 'Color',
               'Used', 'Transmission', 'County']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, NUMERIC),
    ('cat', categorical_transformer, CATEGORICAL)
])

### Define Models

In [ ]:
# Baseline
mlr_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# Untuned models
rf_untuned = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=RANDOM_STATE))
])

xgb_untuned = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=RANDOM_STATE
    ))
])

### Tune Random Forest

In [ ]:
rf_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20],
    'model__min_samples_leaf': [1, 2]
}

rf_search = GridSearchCV(
    rf_untuned,
    rf_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

rf_tuned = rf_search.best_estimator_

print('Best RF params:')
print(rf_search.best_params_)

### Tune XGBoost

In [ ]:
xgb_grid = {
    'model__n_estimators': [200, 300],
    'model__max_depth': [4, 6],
    'model__learning_rate': [0.03, 0.05],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    xgb_untuned,
    xgb_grid,
    n_iter=6,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_search.fit(X_train, y_train)

xgb_tuned = xgb_search.best_estimator_

print('Best XGB params:')
print(xgb_search.best_params_)

### Cross-Validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in [
    ('MLR', mlr_model),
    ('RF Tuned', rf_tuned),
    ('XGB Tuned', xgb_tuned)
]:

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=kf,
        scoring='neg_root_mean_squared_error'
    )

    print(f'{name} CV RMSE: {-scores.mean():.4f}')

## 9. Evaluation

### Evaluation Function

In [ ]:
def evaluate_model(name, model):
    model.fit(X_train, y_train)

    pred_log = model.predict(X_test)

    pred = np.expm1(pred_log)
    actual = np.expm1(y_test)

    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    r2 = r2_score(actual, pred)

    return {
        'Model': name,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2,
        'Predictions': pred
    }

### Compare All Models

In [ ]:
results = []

models = [
    ('MLR', mlr_model),
    ('RF_Untuned', rf_untuned),
    ('RF_Tuned', rf_tuned),
    ('XGB_Untuned', xgb_untuned),
    ('XGB_Tuned', xgb_tuned)
]

for name, model in models:
    res = evaluate_model(name, model)
    results.append(res)

results_df = pd.DataFrame([{
    'Model': r['Model'],
    'RMSE': r['RMSE'],
    'MAPE': r['MAPE'],
    'R2': r['R2']
} for r in results])

results_df = results_df.sort_values('MAPE')

print(results_df)

### Save Results Table

In [ ]:
results_df.to_csv('model_comparison_results.csv', index=False)
print('Results saved.')

## 10. Rare Segment Evaluation

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = dict(models)[best_model_name]

best_model.fit(X_train, y_train)

best_pred = np.expm1(best_model.predict(X_test))

segment_df = test_df.copy()
segment_df['Actual'] = np.expm1(y_test)
segment_df['Predicted'] = best_pred

segment_results = segment_df.groupby('RareSegment').apply(
    lambda g: np.mean(np.abs(
        (g['Actual'] - g['Predicted']) / g['Actual']
    )) * 100
)

print('\nMAPE by segment:')
print(segment_results)

## 11. Feature Importance

### Random Forest Feature Importance

In [ ]:
rf_estimator = rf_tuned.named_steps['model']

feature_names = rf_tuned.named_steps[
    'preprocessor'
].get_feature_names_out()

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_estimator.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance_df.head(15))

plt.figure(figsize=(8,6))
sns.barplot(
    data=importance_df.head(15),
    x='Importance',
    y='Feature'
)
plt.title('Top 15 Important Features')
plt.show()

## 12. SHAP Explainability

In [ ]:
X_train_transformed = rf_tuned.named_steps[
    'preprocessor'
].transform(X_train)

sample_size = min(500, X_train_transformed.shape[0])
X_sample = X_train_transformed[:sample_size]

start = time.time()

explainer = shap.TreeExplainer(rf_estimator)
shap_values = explainer.shap_values(X_sample)

print(f'SHAP completed in {time.time() - start:.2f} seconds')

shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=feature_names,
    max_display=15
)

## 13. Final Plot: Actual vs Predicted

In [ ]:
actual_prices = np.expm1(y_test)

plt.figure(figsize=(6,6))
plt.scatter(actual_prices, best_pred, alpha=0.5)

plt.plot(
    [actual_prices.min(), actual_prices.max()],
    [actual_prices.min(), actual_prices.max()],
    'r--'
)

plt.xlabel('Actual Price (KES)')
plt.ylabel('Predicted Price (KES)')
plt.title(f'{best_model_name}: Actual vs Predicted')
plt.show()